In [ ]:
import numpy as np
import polars as pl
from scipy.optimize import curve_fit
import glob
from concurrent.futures import ProcessPoolExecutor, as_completed
from polars_analysis import *

In [ ]:
PARAMS_CSV   = "Data/Radium2705/fitted_params.csv"
INTEGRAL_CSV = "Data/Radium2705/integrals.csv"
file_pattern = "Data/Radium2705/acq*.csv"

### Stage 1 Fitting all Files

In [ ]:
all_files = glob.glob(file_pattern)
if not all_files:
    print("No files found.")
else:
    rows      = []
    discarded = 0

    with ProcessPoolExecutor() as executor:
        futures = {executor.submit(fit_single_file, fp): fp for fp in all_files}
        for i, future in enumerate(as_completed(futures), 1):
            result = future.result()
            if result is not None:
                rows.append(result)
            else:
                discarded += 1
            if i % 10 == 0:
                print(f"  {i}/{len(all_files)} — kept {len(rows)}, "
                        f"discarded {discarded}", end='\r')

### Stage 2 Computing Integrals